In [ ]:
import os
import json
import numpy as np
import pandas as pd
from joblib import Parallel, delayed

from hdbscan import HDBSCAN
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import seaborn as sns
import matplotlib.pyplot as plt

from utils.functions import load_yaml
from utils.graphfunction import load_graph, get_uniprot_from_nodes, get_res_from_nodes
from data.augmentation import ss_aug
from data.scaler import sin_cos_transform, pca_transform

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Load Configuration
config = load_yaml("../config/RRGCL.yaml")
DATABASE = config.DATABASE

# Load Residue-Level Data
residue_df = pd.read_csv(f'{DATABASE}/merged_feature_data_v041226.csv')

# Load Neighbor-Level Data (Previously pre-computed and saved)
neighbor_df = pd.read_csv('../data/proc_data/ngb_avg_feat.csv')

# Load Graph Data
finalG = load_graph((f"{DATABASE}/cleaned_weighted_graph_041226.pkl"))

## 1. Feature Preprocessing Pipeline

In [ ]:
def preprocess_features(df, selected_ss_cols, pca_components=2, num_ref=10):
    '''
    Extracts and scales AAindex1 features using PCA.
    Extracts and augments secondary structure features.
    Transforms angular SS features into sin/cos components.
    '''
    # 1. AAindex1 Extraction & PCA
    aa1_cols = [col for col in df.columns if col.startswith('aa1')]
    AA1 = df[['node_id'] + aa1_cols].copy()
    if 'aa1' in AA1.columns:
        AA1.drop('aa1', axis=1, inplace=True)
    
    pca_cols = [c for c in aa1_cols if c != 'aa1']
    pca_aa1_df, pca = pca_transform(AA1, pca_cols, pca_components)
    
    # 2. Secondary Structure Extraction & Augmentation
    ss_feat_df = df[selected_ss_cols].copy()
    only_ss_cols = [col for col in selected_ss_cols if col != 'node_id']
    aug_ss_df = ss_aug(ss_feat_df, only_ss_cols, num_ref=num_ref)
    
    # 3. Angular Feature Transformation (sin/cos)
    for angle_col in ['dssp_phi', 'dssp_psi', 'dssp_alpha']:
        if angle_col in selected_ss_cols:
            aug_ss_df = sin_cos_transform(aug_ss_df, angle_col, 'sin')
            aug_ss_df = sin_cos_transform(aug_ss_df, angle_col, 'cos')
            aug_ss_df.drop([angle_col], axis=1, inplace=True)
            
    # Merge both feature sets
    merged_features = pd.merge(aug_ss_df, pca_aa1_df, on='node_id', how='left')
    return merged_features, pca

## 2. Clustering & Grid Search Functions

In [ ]:
def evaluate_hdbscan(mcs, data):
    model = HDBSCAN(min_cluster_size=mcs)
    labels = model.fit_predict(data)
    
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    noise_count = np.sum(labels < 0)
    return n_clusters, noise_count

def run_grid_search(X_scaled, mcs_range, n_jobs=10):
    results = Parallel(n_jobs=n_jobs)(
        delayed(evaluate_hdbscan)(mcs, X_scaled) for mcs in mcs_range
    )
    n_clusters_list, noise_counts = zip(*results)
    return list(n_clusters_list), list(noise_counts)

def plot_hdbscan_results(mcs_range, n_clusters_list, noise_counts, title_suffix=""):
    fig, ax1 = plt.subplots(figsize=(12, 6))

    ax1.set_xlabel('min_cluster_size')
    ax1.set_ylabel('Number of Clusters', color='tab:blue')
    ax1.plot(mcs_range, n_clusters_list, marker='o', color='tab:blue', label='Clusters')
    ax1.set_yscale('log')
    ax1.tick_params(axis='y', labelcolor='tab:blue')

    ax2 = ax1.twinx()
    ax2.set_ylabel('Number of Noise Points', color='tab:red')
    ax2.plot(mcs_range, noise_counts, marker='s', linestyle='--', color='tab:red', label='Noise')
    ax2.set_yscale('log')
    ax2.tick_params(axis='y', labelcolor='tab:red')

    plt.title(f'HDBSCAN: Clusters and Noise vs. min_cluster_size {title_suffix}')
    plt.xticks(list(mcs_range))
    plt.grid(True, axis='x', linestyle=':', alpha=0.7)
    plt.show()

## 3. Residue-Level Clustering

In [ ]:
# Preprocess
ss_cols_residue = ['node_id', 'rel_sasa', 'depth', 'hse_up', 'hse_down', 'dssp_phi', 'dssp_psi', 'dssp_alpha', 'dssp_TCO', 'dssp_accessibility']
residue_features_df, residue_pca = preprocess_features(residue_df, ss_cols_residue, num_ref=10)
print(f"AAindex PCA Explained Variance: {residue_pca.explained_variance_ratio_}")

# Feature Selection
aa1_pca_cols = [c for c in residue_features_df.columns if c.startswith('aa1_PCA')]
sim_cols = aa1_pca_cols + ['rel_sasa', 'depth', 'hse_up', 'hse_down', 'dssp_phi_sin', 'dssp_alpha_sin', 'dssp_phi_cos', 'dssp_alpha_cos']
residue_final_df = residue_features_df[sim_cols]

# Optional: Correlation Heatmap
corr = residue_final_df.select_dtypes(include=['number']).corr()
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title('Residue Feature Correlation Heatmap')
plt.show()

In [ ]:
# Scaling
residue_X = residue_final_df.values
scaler = StandardScaler()
residue_X_scaled = scaler.fit_transform(residue_X)                                                                                                  

# Grid Search
residue_mcs_range = range(10, 21)
residue_clusters, residue_noise = run_grid_search(residue_X_scaled, residue_mcs_range, n_jobs=10)
plot_hdbscan_results(residue_mcs_range, residue_clusters, residue_noise, "(Residue)")

In [ ]:
# Extract optimal clusters (e.g., mcs=13 based on earlier notebooks)
residue_model = HDBSCAN(min_cluster_size=13)
residue_features_df['cluster'] = residue_model.fit_predict(residue_X_scaled)

print(f"Noise points (-1): {len(residue_features_df[residue_features_df['cluster'] < 0])}")
print("Cluster distribution:")
print(residue_features_df['cluster'].value_counts())

## 4. Neighbor-Level Clustering

In [ ]:
# Preprocess
ss_cols_neighbor = ['node_id', 'rel_sasa', 'depth', 'hse_up', 'hse_down', 'dssp_phi', 'dssp_psi', 'dssp_alpha', 'dssp_TCO', 'dssp_accessibility']
neighbor_features_df, neighbor_pca = preprocess_features(neighbor_df, ss_cols_neighbor, num_ref=5)

# Feature Selection
neighbor_final_df = neighbor_features_df[sim_cols]

# Scaling
neighbor_X = neighbor_final_df.values
neighbor_X_scaled = scaler.fit_transform(neighbor_X)

# Grid Search
neighbor_mcs_range = range(5, 21)
neighbor_clusters, neighbor_noise = run_grid_search(neighbor_X_scaled, neighbor_mcs_range, n_jobs=8)
plot_hdbscan_results(neighbor_mcs_range, neighbor_clusters, neighbor_noise, "(Neighbor)")

In [ ]:
# Extract optimal clusters (e.g., mcs=8 based on earlier notebooks)
neighbor_model = HDBSCAN(min_cluster_size=8)
neighbor_features_df['cluster'] = neighbor_model.fit_predict(neighbor_X_scaled)

print(f"Noise points (-1): {len(neighbor_features_df[neighbor_features_df['cluster'] < 0])}")
print("Cluster distribution:")
print(neighbor_features_df['cluster'].value_counts())

## 5. Cluster Content Distribution

In [ ]:
# residue_features_df = pd.read_csv()
# neighbor_features_df = pd.read_csv()

In [ ]:
def plot_cluster_distribution(cluster_df, target_col, grid_rows=4, grid_cols=3):
    clusters = cluster_df['cluster'].unique()
    clusters.sort()
    
    fig, axes = plt.subplots(grid_rows, grid_cols, figsize=(grid_cols * 4, grid_rows * 3))
    axes = axes.ravel()
    
    for i, clt in enumerate(clusters[:grid_rows * grid_cols]):
        data = cluster_df[cluster_df['cluster'] == clt][target_col]
        counts = data.value_counts()
        sns.barplot(x=counts.index, y=counts.values, ax=axes[i], palette='viridis')
        axes[i].set_title(f"Cluster {clt}")
        axes[i].tick_params(axis='x', rotation=45)
        
    plt.tight_layout()
    plt.show()

# Add mapping features
residue_features_df['residue'] = residue_features_df['node_id'].map(get_res_from_nodes)
residue_features_df['uniprot'] = residue_features_df['node_id'].map(get_uniprot_from_nodes)

neighbor_features_df['residue'] = neighbor_features_df['node_id'].map(get_res_from_nodes)
neighbor_features_df['uniprot'] = neighbor_features_df['node_id'].map(get_uniprot_from_nodes)

In [ ]:
print("Residue-Level Clustering: Residue Types")
plot_cluster_distribution(residue_features_df, 'residue', grid_rows=3, grid_cols=4)

print("Neighbor-Level Clustering: Residue Types")
plot_cluster_distribution(neighbor_features_df, 'residue', grid_rows=3, grid_cols=4)

## 6. Miscellaneous (Saving & External Loading)

In [ ]:
# Example saving to CSV
# residue_features_df.to_csv('node_cluster_comp2_minmax_minC13.csv', index=False)
# neighbor_features_df.to_csv('ngb_cluster_comp2_minmax_minC8_hop1.csv', index=False)

# AlphaMissense / PSSM / HMM examples
# with open(f"{DATABASE}/am_features.json", 'rb') as f:
#     am = json.load(f)
# with open('proc_data/pssm_features.json') as f:
#     pssm = json.load(f)
